# Lección 2.1 — Tipos de datos: no todos los ingredientes se cocinan igual

> **Módulo 2 · Los datos: el combustible** · Nivel 1 — Cero
> ⏱️ Tiempo estimado: 70–90 min (lección con ejercicios intercalados ≈ 50 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 1.2 (vocabulario: dato, característica, etiqueta)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Distinguir datos **numéricos**, **categóricos**, de **texto**, de **imagen** y **series de tiempo**.
- Detectar columnas que *parecen* números pero en realidad son categorías (y al revés).
- Explicar por qué una computadora necesita convertir texto e imágenes en números.
- Reconocer cuándo el **orden** de los datos importa (series de tiempo).

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.feature_extraction.text import CountVectorizer

plt.rcParams["figure.figsize"] = (8, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: la cocina y sus ingredientes

Imagina que vas a cocinar. Tienes **papas, leche, sal y una receta escrita en un papel**.
- Las papas se **pelan y cortan**.
- La leche se **mide** en mililitros.
- La sal se **cuenta** en pizcas.
- La receta se **lee**.

Cada ingrediente necesita un **tratamiento distinto**. Si intentas "medir en mililitros" una papa, o "pelar" la leche, el resultado es un desastre.

Con los datos pasa lo mismo. Un modelo de Machine Learning solo entiende **números**, pero no todos los datos *son* números, y aun los que lo parecen no siempre se pueden tratar como tales. **Antes de entrenar cualquier cosa, tienes que saber qué tipo de ingrediente tienes en cada columna.**

### ✋ Ejercicio 1 — Calentamiento
Una clínica guarda esta información de cada paciente. Para cada dato, decide si es un **número que se puede medir** (tiene sentido sumarlo o promediarlo) o una **etiqueta / categoría** (sirve para agrupar, no para hacer cuentas):

1. Peso (kg) · 2. Tipo de sangre · 3. Número de visitas en el año · 4. Ciudad de residencia · 5. Número de teléfono

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Número** (peso promedio tiene sentido). 2. **Categoría** (A, B, AB, O). 3. **Número** (se cuenta). 4. **Categoría**.
5. **Categoría… aunque esté escrito con dígitos.** El "promedio de los teléfonos" no significa nada. Esta trampa es la más común de todas y la verás varias veces en esta lección.
</details>

---
## 2. 📖 El concepto: la tabla y sus tipos de columnas

La forma más común de datos en ML es una **tabla** (Lección 1.2): cada **fila** es un ejemplo y cada **columna** es una característica (*feature*).

| Tipo | Subtipo | Ejemplo | ¿Qué cuentas tiene sentido hacer? |
|---|---|---|---|
| **Numérico** | Continuo | precio, temperatura, peso | Promediar, restar, comparar |
| | Discreto (conteo) | número de habitaciones, de hijos | Promediar, comparar |
| **Categórico** | Nominal (sin orden) | color, ciudad, tipo de sangre | Solo contar cuántos hay de cada uno |
| | Ordinal (con orden) | talla S/M/L, nivel educativo, estrellas ★ | Ordenar, pero **no** restar ("L − S" no es un número) |
| | Binario | sí/no, tiene garaje | Contar, proporción |
| **Fecha / hora** | — | fecha de compra | Restar fechas, extraer día, mes, hora |
| **Texto libre** | — | una reseña, un correo | Hay que convertirlo antes (Sección 4) |

Veámoslo con datos. Una inmobiliaria tiene estos departamentos en renta:

In [ ]:
deptos = pd.DataFrame({
    "renta_mensual":  [12_500, 18_000, 9_800, 25_000, 15_200, 11_000],
    "m2":             [55.0, 80.5, 42.0, 120.0, 68.0, 50.0],
    "habitaciones":   [1, 2, 1, 3, 2, 1],
    "colonia":        ["Centro", "Roma", "Centro", "Polanco", "Roma", "Doctores"],
    "codigo_postal":  [6000, 6700, 6000, 11560, 6700, 6720],
    "estado":         ["Bueno", "Excelente", "Regular", "Excelente", "Bueno", "Regular"],
    "acepta_mascotas": [True, False, True, True, False, True],
    "publicado":      ["2024-03-01", "2024-03-05", "2024-03-07", "2024-03-10", "2024-03-12", "2024-03-15"],
    "descripcion":    ["Luminoso y céntrico", "Con balcón y vista", "Pequeño pero cómodo",
                       "Lujo con alberca", "Remodelado, con balcón", "Cerca del metro"],
})
deptos

Pandas (la librería de tablas) le asigna un **tipo** (`dtype`) a cada columna. Pero lo adivina mirando *cómo está escrito* el dato, no *qué significa*.

### 🔮 Ejercicio 2 — ¿Qué cree la computadora?
Antes de ejecutar: ¿qué tipo crees que le asignará pandas a `codigo_postal`? ¿Y a `publicado`? ¿Son esos los tipos **correctos** según su significado?

✍️ *Mi apuesta:*

In [ ]:
deptos.dtypes

<details><summary><b>👀 Ver respuesta</b></summary>

- `codigo_postal` → `int64` (**número entero**), porque está escrito con dígitos. Pero un código postal es una **categoría**: el 11560 no es "el doble" del 6000.
- `publicado` → `str` (**texto**; en versiones antiguas de pandas aparece como `object`), porque pandas no sabe que eso es una fecha. Hay que decírselo.
- `estado` → `str` (texto), y pandas no sabe que "Regular < Bueno < Excelente".

**Lección:** el tipo que asigna la computadora es solo una suposición. **Tú** decides el tipo según lo que el dato significa.
</details>

### 📖 La prueba del promedio
Una regla práctica para saber si algo es *de verdad* un número: **¿tiene sentido su promedio?**

In [ ]:
print("Promedio de renta         :", deptos["renta_mensual"].mean())
print("Promedio de código postal :", deptos["codigo_postal"].mean())

La renta promedio es útil. El "código postal promedio" (7,280) es un número que **no corresponde a ningún lugar**: es basura con apariencia de dato.

### ✋ Ejercicio 3 — Clasifica las columnas
Completa la tabla para las 9 columnas de `deptos`: numérico (continuo/discreto), categórico (nominal/ordinal/binario), fecha o texto.

✍️ *Tu tabla:* renta_mensual → ___ · m2 → ___ · habitaciones → ___ · colonia → ___ · codigo_postal → ___ · estado → ___ · acepta_mascotas → ___ · publicado → ___ · descripcion → ___

<details><summary><b>👀 Ver respuesta</b></summary>

| Columna | Tipo |
|---|---|
| `renta_mensual` | Numérico continuo (es la **etiqueta** si queremos predecir la renta) |
| `m2` | Numérico continuo |
| `habitaciones` | Numérico discreto |
| `colonia` | Categórico nominal |
| `codigo_postal` | Categórico nominal (¡aunque tenga dígitos!) |
| `estado` | Categórico **ordinal** (Regular < Bueno < Excelente) |
| `acepta_mascotas` | Categórico binario |
| `publicado` | Fecha |
| `descripcion` | Texto libre |
</details>

---
## 3. 📖 Nominal vs. ordinal: ¿hay un orden?

Las categorías se parecen, pero hay una diferencia clave:
- **Nominal:** no hay orden. "Roma" no es mayor ni menor que "Centro".
- **Ordinal:** sí hay orden. "Excelente" es mejor que "Bueno", y "Bueno" mejor que "Regular". Pero **la distancia entre escalones no está definida**: ¿de "Regular" a "Bueno" hay lo mismo que de "Bueno" a "Excelente"? No lo sabemos.

¿Por qué importa? Porque al convertir categorías a números (Lección 2.3) podemos **inventar un orden que no existe**. Mira este error clásico:

In [ ]:
# Alguien convierte los colores de una camiseta a números "porque el modelo solo entiende números"
codigo_color = {"Rojo": 1, "Verde": 2, "Azul": 3}
ventas = pd.Series(["Rojo", "Azul", "Rojo", "Azul"])

promedio = ventas.map(codigo_color).mean()
print("Color 'promedio' vendido:", promedio, "→", [c for c, n in codigo_color.items() if n == promedio])

### 🔮 Ejercicio 4 — ¿Qué salió mal?
Se vendieron 2 camisetas rojas y 2 azules, y **ninguna verde**. ¿Por qué el resultado dice "Verde"?

✍️ *Tu explicación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Al poner Rojo = 1, Verde = 2, Azul = 3, le dijimos a la computadora que **el verde está "entre" el rojo y el azul**, y que el azul es "3 veces" el rojo. Es un orden inventado. Para categorías nominales hay que usar otra codificación (*one-hot*, Lección 2.3) que no invente orden.

Con categorías **ordinales** (talla S/M/L), en cambio, numerarlas en orden sí conserva información útil.
</details>

### ✋ Ejercicio 5 — ¿Nominal u ordinal?
1. Talla de ropa (S, M, L, XL) · 2. Marca de celular · 3. Nivel de satisfacción (😡 😐 🙂 😍) · 4. Método de pago · 5. Calificación crediticia (AAA, AA, A, BBB…) · 6. Idioma

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**Ordinales:** 1, 3 y 5 (tienen un orden natural). **Nominales:** 2, 4 y 6.
</details>

---
## 4. 📖 Texto: convertir palabras en números

Un modelo no puede "leer" la frase *"Luminoso y céntrico"*. La forma más simple de convertir texto en números es la **bolsa de palabras** (*bag of words*): se arma un **vocabulario** con todas las palabras que aparecen y, para cada texto, se **cuenta** cuántas veces aparece cada una. El orden de las palabras se pierde (por eso es una "bolsa").

### 🔮 Ejercicio 6 — ¿Cuántas columnas?
Vamos a convertir 3 reseñas de restaurantes en una tabla de conteos. Antes de ejecutar: ¿cuántas **columnas** tendrá la tabla? (Pista: cuenta las palabras *distintas*.)

- "comida rica"
- "comida fría y cara"
- "rica y barata"

✍️ *Mi apuesta:*

In [ ]:
resenas = ["comida rica", "comida fría y cara", "rica y barata"]

vectorizador = CountVectorizer(token_pattern=r"(?u)\b\w+\b")   # cuenta todas las palabras, incluso las de 1 letra
conteos = vectorizador.fit_transform(resenas)

pd.DataFrame(conteos.toarray(), columns=vectorizador.get_feature_names_out(), index=resenas)

<details><summary><b>👀 Ver respuesta</b></summary>

**6 columnas**: `barata`, `cara`, `comida`, `fría`, `rica`, `y`. Cada reseña es ahora una fila de números, y el texto ya se puede usar en un modelo.

Fíjate en dos problemas:
- Con miles de reseñas el vocabulario tiene **miles de columnas**, casi todas en cero (verás por qué eso es un problema en la Lección 2.8).
- "No está rica" y "está rica" quedan casi iguales: la bolsa de palabras **pierde el orden**. Los modelos de lenguaje modernos resuelven esto con *embeddings* y atención (Módulos 11 y 12).
</details>

---
## 5. 📖 Imágenes: una cuadrícula de números

Para una computadora, una imagen en blanco y negro es una **cuadrícula de números**: cada casilla (píxel) dice qué tan oscura es. Una imagen a color tiene **tres** cuadrículas: rojo, verde y azul.

Scikit-learn trae un conjunto de 1,797 dígitos escritos a mano. Veamos uno.

### 🔮 Ejercicio 7 — ¿Cuántos números describen una imagen?
Cada dígito es una imagen de **8 × 8** píxeles. ¿Cuántas características (columnas) tendrá cada ejemplo en la tabla?

✍️ *Mi apuesta:*

In [ ]:
digitos = load_digits()
print("Forma de la tabla:", digitos.data.shape, "→ (ejemplos, características)")

imagen = digitos.images[0]
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(imagen, cmap="gray_r"); ax[0].set_title(f"Lo que ves tú: un {digitos.target[0]}")
ax[1].imshow(imagen, cmap="gray_r", alpha=.3)
for (i, j), v in np.ndenumerate(imagen):
    ax[1].text(j, i, int(v), ha="center", va="center", fontsize=9)
ax[1].set_title("Lo que ve la computadora")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

**64 características** (8 × 8). La tabla tiene 1,797 filas y 64 columnas: cada píxel es una columna, con valores de 0 (blanco) a 16 (negro).

Ahora piensa en una foto de celular de 12 megapíxeles a color: son **36 millones** de números por imagen. Por eso las imágenes necesitan modelos especiales (redes convolucionales, Módulo 10) que aprovechan que los píxeles vecinos están relacionados.
</details>

---
## 6. 📖 Series de tiempo: cuando el orden importa

En una tabla normal puedes **revolver las filas** y nada cambia: cada cliente es independiente. Pero en una **serie de tiempo** (ventas diarias, temperatura por hora, precio de una acción), cada dato depende de los anteriores. **El orden es parte de la información.**

Veamos las ventas diarias de una cafetería durante 8 semanas:

In [ ]:
rng = np.random.default_rng(1)
dias = pd.date_range("2024-01-01", periods=56, freq="D")          # empieza en lunes
patron_semanal = np.array([60, 55, 58, 62, 80, 120, 110])        # lun ... dom
ventas_cafe = pd.Series(np.tile(patron_semanal, 8) + np.arange(56) * 0.8 + rng.normal(0, 6, 56),
                        index=dias).round()

fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharey=True)
ax[0].plot(ventas_cafe.values, "o-", ms=3); ax[0].set_title("Ventas en orden")
ax[1].plot(ventas_cafe.sample(frac=1, random_state=0).values, "o-", ms=3, color="gray")
ax[1].set_title("Las mismas ventas, revueltas")
plt.tight_layout(); plt.show()

### ✋ Ejercicio 8 — Lee la serie
Mirando la gráfica de arriba (la ordenada):
1. ¿Ves algún patrón que se repite? ¿Cada cuánto?
2. ¿Las ventas suben, bajan o se mantienen a lo largo de las 8 semanas?
3. ¿Qué pasó con esos patrones en la gráfica revuelta?

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Sí: un pico **cada 7 días** (los fines de semana se vende el doble). Eso se llama **estacionalidad**.
2. Suben poco a poco: hay una **tendencia** creciente.
3. **Desaparecieron.** Los números son exactamente los mismos, pero sin el orden ya no hay patrón ni tendencia.

Consecuencia práctica muy importante: con series de tiempo **no puedes revolver los datos al dividirlos** en entrenamiento y prueba. Hay que entrenar con el pasado y evaluar con el futuro (Lecciones 2.5 y 2.6).
</details>

### 📖 La fecha como fuente de características
Una fecha como `2024-01-06` no le dice mucho a un modelo. Pero de ella se pueden **extraer** datos muy útiles: día de la semana, mes, si es fin de semana, si es feriado… (lo practicarás en la Lección 2.4).

In [ ]:
tabla_cafe = pd.DataFrame({"ventas": ventas_cafe})
tabla_cafe["dia_semana"] = tabla_cafe.index.day_name()
tabla_cafe["fin_de_semana"] = tabla_cafe.index.day_of_week >= 5

tabla_cafe.groupby("fin_de_semana")["ventas"].mean().round(1)

### 🔮 Ejercicio 9
¿Cuántas veces más vende, en promedio, la cafetería en fin de semana que entre semana? Calcúlalo con el resultado de arriba.

✍️ *Mi cálculo:*

<details><summary><b>👀 Ver respuesta</b></summary>

Alrededor de **1.6 veces** (≈139 vs. ≈84 ventas por día). Una sola columna creada a partir de la fecha (`fin_de_semana`) ya separa claramente los días buenos de los normales.
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué hacer |
|---|---|---|
| Un "promedio" que no significa nada (código postal, teléfono, ID) | Una categoría guardada como número | Convertirla a texto/categoría |
| El modelo cree que "Azul > Rojo" | Categorías nominales numeradas 1, 2, 3 | Usar *one-hot* (Lección 2.3) |
| Una fecha guardada como texto | Pandas no la reconoció | `pd.to_datetime` y extraer partes (Lección 2.4) |
| Números guardados como texto (`"1,200"`, `"$500"`) | Comas, símbolos de moneda | Limpiar y convertir (Lección 2.2) |
| Resultados espectaculares con una serie de tiempo | Se revolvieron los datos y el modelo "vio el futuro" | División temporal (Lecciones 2.5 y 2.6) |

### 🔮 Ejercicio 10 — El precio que no suma
Una tienda exportó sus precios así. Antes de ejecutar: ¿qué crees que devolverá `.sum()`?

In [ ]:
precios = pd.Series(["$1,200", "$350", "$80"])
print("Tipo:", precios.dtype)
print("Suma:", precios.sum())

<details><summary><b>👀 Ver respuesta</b></summary>

No suma: **pega los textos** (`$1,200$350$80`) porque la columna es de tipo `str` (texto; `object` en versiones antiguas de pandas). Hay que quitar `$` y `,` y convertir a número:
```python
precios.str.replace("[$,]", "", regex=True).astype(float).sum()   # 1630.0
```
Es un problema de **calidad de datos**, el tema de la Lección 2.2.
</details>

---
## 8. 🧠 Resumen en 5 puntos
1. Un modelo solo entiende **números**, pero no todos los datos son números: hay que identificar el **tipo** de cada columna.
2. **Numéricos** (continuos o conteos) se pueden promediar; **categóricos** (nominales, ordinales, binarios) solo se cuentan o se ordenan.
3. Que algo esté escrito con dígitos **no lo convierte en número**: códigos postales, teléfonos e IDs son categorías. Usa la **prueba del promedio**.
4. **Texto** e **imágenes** se convierten en tablas de números (bolsa de palabras, píxeles), a menudo con **muchísimas** columnas.
5. En las **series de tiempo** el orden es información: no se pueden revolver.

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** La columna `numero_de_empleado` tiene valores como 10234, 10235, 10236… ¿Qué tipo de dato es?
- a) Numérico continuo · b) Numérico discreto · c) Categórico (un identificador) · d) Fecha

**2.** ¿Cuál de estas columnas es **ordinal**?
- a) País de nacimiento · b) Nivel de estudios (primaria, secundaria, universidad) · c) Color de ojos · d) Correo electrónico

**3.** Conviertes 1,000 correos electrónicos con bolsa de palabras y el vocabulario tiene 8,000 palabras. ¿Qué forma tiene la tabla?
- a) 8,000 × 1,000 · b) 1,000 × 8,000 · c) 1,000 × 1 · d) 1 × 8,000

**4.** Una foto en blanco y negro de 28 × 28 píxeles, ¿cuántas características tiene?
- a) 28 · b) 56 · c) 784 · d) 2,352

**5.** ¿En cuál de estos casos **no** debes revolver las filas antes de dividir los datos?
- a) Fotos de gatos y perros · b) Encuesta a 500 personas · c) Consumo eléctrico por hora de una ciudad · d) Reseñas de películas

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** Es un identificador: su promedio no significa nada. Además, los identificadores pueden causar **fugas de datos** (Lección 2.6), así que no se usan como características.
2. **b)** Tiene un orden natural. Las demás son nominales (o identificadores, como el correo).
3. **b)** Una fila por correo y una columna por palabra del vocabulario.
4. **c)** 28 × 28 = 784. (La d, 2,352 = 784 × 3, sería a color.)
5. **c)** Es una serie de tiempo: el consumo de una hora depende de las anteriores.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** La columna `edad` es numérica. ¿Se te ocurre una situación en la que convenga tratarla como **categoría ordinal** (por ejemplo, "18–25", "26–40", "41–60", "60+")? ¿Qué ganas y qué pierdes?

**B. Caso de análisis 💼** Un hospital quiere predecir si un paciente volverá a internarse. Tiene: notas del médico (texto), radiografías (imagen), presión arterial medida cada hora durante la internación (serie de tiempo), edad, sexo y código del seguro médico. ¿Qué tipo de dato es cada uno? ¿Cuál crees que será **más difícil** de preparar y por qué?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Conviene cuando la relación con lo que quieres predecir cambia "por tramos" (p. ej., una aseguradora con tarifas por grupo de edad) o cuando necesitas comunicar resultados de forma simple. **Ganas** simplicidad y robustez ante errores pequeños; **pierdes** detalle (un joven de 26 y un adulto de 40 quedan iguales). Se llama **discretizar** y lo verás en la Lección 2.4.

**B.** Notas → **texto**; radiografías → **imagen**; presión cada hora → **serie de tiempo**; edad → **numérico**; sexo → **categórico binario/nominal**; código del seguro → **categórico nominal** (aunque tenga dígitos). Los más difíciles suelen ser el texto (lenguaje médico, abreviaturas, errores) y las imágenes (muchísimos números y modelos especializados). No hay una única respuesta: lo importante es que cada tipo requiere un tratamiento distinto.
</details>

### 9.3 🧩 Práctica integradora: "La tienda en línea"

Una tienda en línea te entrega su tabla de pedidos y te pide revisar que **cada columna tenga el tipo correcto** antes de empezar a modelar.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos**

In [ ]:
pedidos = pd.DataFrame({
    "id_pedido":      [50311, 50312, 50313, 50314, 50315, 50316],
    "fecha":          ["2024-05-03", "2024-05-04", "2024-05-04", "2024-05-06", "2024-05-11", "2024-05-12"],
    "codigo_postal":  ["06700", "44100", "06700", "64000", "44100", "03100"],
    "categoria":      ["Ropa", "Hogar", "Ropa", "Electrónica", "Hogar", "Ropa"],
    "talla":          ["M", None, "S", None, None, "XL"],
    "unidades":       [2, 1, 3, 1, 4, 1],
    "total":          ["$1,198", "$450", "$897", "$5,999", "$1,800", "$799"],
    "calificacion":   [5, 4, 3, 5, 2, 4],          # estrellas que dio el cliente (1 a 5)
    "comentario":     ["Excelente", "Llegó tarde", "Talla pequeña", "Muy buen producto", "Llegó roto", "Ok"],
    "devuelto":       [0, 0, 1, 0, 1, 0],          # 🎯 lo que queremos predecir
})
pedidos.dtypes

**Paso 2 ✋ — Diagnóstico.** Para cada columna escribe (a) qué tipo **asignó pandas** y (b) qué tipo **es de verdad**. Marca las que están mal.

> 📝 Según tu versión de pandas, las columnas de texto aparecen como `str` o como `object`: significan lo mismo.

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver respuesta</b></summary>

| Columna | pandas dice | En realidad es | ¿Bien? |
|---|---|---|---|
| `id_pedido` | int64 | Identificador (no es característica) | ❌ |
| `fecha` | str | Fecha | ❌ |
| `codigo_postal` | str | Categórico nominal | ✅ (bien guardado como texto: conserva el `0` inicial) |
| `categoria` | str | Categórico nominal | ✅ |
| `talla` | str | Categórico **ordinal** (con faltantes: no aplica a Hogar o Electrónica) | ⚠️ falta el orden |
| `unidades` | int64 | Numérico discreto | ✅ |
| `total` | str | Numérico continuo (escrito con `$` y `,`) | ❌ |
| `calificacion` | int64 | Categórico ordinal (1 a 5 estrellas) — también se suele tratar como número | ✅/⚠️ |
| `comentario` | str | Texto libre | ✅ (pero habrá que convertirlo) |
| `devuelto` | int64 | Binario — es la **etiqueta** | ✅ |

¿Te diste cuenta de que `codigo_postal` aquí **sí** está bien? Si se hubiera guardado como número, `06700` se habría convertido en `6700` y perderías información.
</details>

**Paso 3 ✏️ — Arregla la fecha y el total.** Ejecuta la celda y comprueba que ahora `total` suma.

In [ ]:
pedidos["fecha"] = pd.to_datetime(pedidos["fecha"])
pedidos["total"] = pedidos["total"].str.replace("[$,]", "", regex=True).astype(float)

print(pedidos[["fecha", "total"]].dtypes)
print("\nVentas totales:", pedidos["total"].sum())

**Paso 4 ✏️ — Dale orden a la talla.** Una categoría ordinal se puede declarar con su orden. Ejecuta y luego responde: ¿qué devuelve la última línea y por qué es útil?

In [ ]:
orden_tallas = ["S", "M", "L", "XL"]       # ✏️ ¿qué pasa si cambias el orden?
pedidos["talla"] = pd.Categorical(pedidos["talla"], categories=orden_tallas, ordered=True)

print(pedidos["talla"])
print("\n¿Pedidos con talla mayor que M?\n", pedidos["talla"] > "M")

<details><summary><b>👀 Ver respuesta</b></summary>

Devuelve `True` solo para el pedido con talla **XL** (y `False` para los faltantes). Ahora la computadora **sabe** que S < M < L < XL y puede comparar tallas. Si cambias el orden en `orden_tallas`, la comparación cambia: el orden lo decides tú, no la computadora.
</details>

**Paso 5 ✋ — Saca información de la fecha.** Crea una columna `dia_semana` a partir de `fecha` (pista: `.dt.day_name()`, como en la Sección 6) y mira en qué días hubo devoluciones. ¿Qué otras dos características se te ocurren a partir de la fecha?

In [ ]:
pedidos["dia_semana"] = pedidos["fecha"].dt.day_name()    # ✏️ prueba también .dt.month o .dt.dayofweek
pedidos[["fecha", "dia_semana", "devuelto"]]

<details><summary><b>👀 Ver respuesta</b></summary>

Las dos devoluciones fueron un **Saturday** (sábado) y otro **Saturday**: el 4 y el 11 de mayo de 2024. Con 6 pedidos esto **no prueba nada** (es una coincidencia hasta que haya muchos más datos), pero es justo el tipo de pista que un modelo podría usar.

Otras características posibles: `es_fin_de_semana`, `mes`, `dias_hasta_navidad`, `es_quincena` (en muchos países se cobra el 15 y el último día del mes), `es_buen_fin`/`black_friday`. Lo practicarás en serio en la Lección 2.4.
</details>

---
## 📚 Para profundizar
- 📘 **Documentación:** pandas, guía *Categorical data* (cómo declarar categorías y su orden).
- 📺 **Video:** StatQuest, "Data types" y la serie sobre *one-hot encoding* (inglés, con subtítulos).
- 🎮 **Interactivo:** explora los conjuntos de datos de Kaggle y, para cada uno, intenta clasificar sus columnas antes de leer la descripción.

---
⬅️ **Anterior:** [1.7 Proyecto del módulo: Gimnasio "FitVida": ¿quién se va a dar de baja? 🏋️](../m01-que-es-ml/1.7-proyecto-del-modulo.md) · ➡️ **Siguiente:** [2.2 Calidad de datos: faltantes, atípicos, duplicados y ruido](2.2-calidad-de-datos.ipynb)